# Cross-step tx→tx graph — Step 1: build & profile

**Motivation.** The existing pipeline uses bipartite addr↔tx edges only. Phase 1 trajectory features summarize each wallet's history into vectors but **lose the tx-pair structure** — which specific past tx `T₁` passed value to which present tx `T₂`. The literature gap on Elliptic++ under the inductive + strictly causal regime is that no model in this repo (and few in the literature) sees the actual money-flow tx→tx graph.

**This step.** Construct the cross-timestep tx→tx graph induced by shared wallets and characterize its statistics, *before* committing to any feature engineering or GNN. Decide the hub-pruning threshold that keeps the edge set tractable without throwing away useful signal.

## Edge construction rule

For every wallet `w`, sort its incident txs by time. For every pair `(T_i, T_j)` with `t(T_i) < t(T_j)` and `T_j ≠ T_i`, emit a **directed temporal edge** `T_i → T_j` with attributes:
- `wallet`: shared wallet identity
- `dt = t(T_j) − t(T_i)` (≥ 1 by construction)
- `role ∈ {out→in, out→out, in→in, in→out}`
  - `out→in` is the actual money-flow direction (T_i sent BTC to w, w then sent to T_j as input)
  - the other roles are still informative (co-occurrence of wallet across txs)

Causality is automatic — `t(T_i) < t(T_j)` strictly. At classification time for T at `t`, only edges where T is the destination and `t(source) < t` matter; edges where T is the source are not used to score T (they would be ≤ t but reflect future co-occurrences if not pruned). This is enforced downstream when we build per-tx features.

## Hub-pruning rationale

Exchange / mixer wallets touch thousands of txs. A wallet with `k` incident txs contributes `O(k²)` edges. With ~14K wallets having ≥ 5 incident txs and the heaviest hubs in the 10K-incidence range, the unpruned edge count is in the 50M+ range. We need a per-wallet degree cap. This notebook profiles the trade-off.

## What we read
- `transactions_data/txs_features.txt`, `txs_classes.txt`
- `actors_data/AddrTx_edgelist.txt`, `TxAddr_edgelist.txt`

No `wallets_features.txt`, no `AddrAddr_edgelist.txt` (leak-prone). No new data sources.

## What we write
Nothing yet — this is a profiling notebook. Outputs go to print statements and matplotlib (no files saved). Step 2 will compute features.

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

ROOT = Path.cwd()
while not (ROOT / "transactions_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR      = ROOT / "actors_data"
assert TRANSACTIONS_DIR.exists() and WALLETS_DIR.exists()
print(f"repo root: {ROOT}")

TRAIN_END     = 34
N_TIME_STEPS  = 49
RANDOM_SEED   = 175
np.random.seed(RANDOM_SEED)

repo root: /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project


In [2]:
print("Loading transactions...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.txt")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.txt")
tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
merged = tx_features_df[["txId","Time step"]].merge(
    tx_classes_df[["txId","label"]], on="txId", how="left"
)
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
print(f"  N_tx={N_tx:,}  illicit={int((tx_label==1).sum()):,}  "
      f"licit={int((tx_label==0).sum()):,}  unknown={int((tx_label==-1).sum()):,}")

print("\nLoading bipartite addr↔tx edges...")
addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")  # input wallet -> tx
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")  # tx -> output wallet
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) |
                      set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a: i for i, a in enumerate(wallet_addrs)}
N_w = len(wallet_addr_to_idx)
print(f"  unique wallets: {N_w:,}")

# Per-tx incident wallets, split by side
addr_tx_df["w"]  = addr_tx_df["input_address"].map(wallet_addr_to_idx)
addr_tx_df["tx"] = addr_tx_df["txId"].map(tx_id_to_idx)
addr_tx_df = addr_tx_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
tx_addr_df["w"]  = tx_addr_df["output_address"].map(wallet_addr_to_idx)
tx_addr_df["tx"] = tx_addr_df["txId"].map(tx_id_to_idx)
tx_addr_df = tx_addr_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})

incident_in_w  = defaultdict(list)  # tx -> [wallets that fed it]
incident_out_w = defaultdict(list)  # tx -> [wallets it paid]
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values):
    incident_in_w[int(tx)].append(int(w))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values):
    incident_out_w[int(tx)].append(int(w))

# Per-wallet incident txs, split by side and chronologically sorted
wallet_in_txs  = defaultdict(list)   # wallet w -> list of (tx, t) where w was input  to tx
wallet_out_txs = defaultdict(list)   # wallet w -> list of (tx, t) where w was output of tx
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values):
    wallet_in_txs[int(w)].append((int(tx), int(tx_time[tx])))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values):
    wallet_out_txs[int(w)].append((int(tx), int(tx_time[tx])))
for w in wallet_in_txs:  wallet_in_txs[w].sort(key=lambda r: r[1])
for w in wallet_out_txs: wallet_out_txs[w].sort(key=lambda r: r[1])

print(f"  wallets with input incidence:  {len(wallet_in_txs):,}")
print(f"  wallets with output incidence: {len(wallet_out_txs):,}")

Loading transactions...


  N_tx=203,769  illicit=4,545  licit=42,019  unknown=157,205

Loading bipartite addr↔tx edges...


  unique wallets: 822,942


  wallets with input incidence:  400,212
  wallets with output incidence: 641,043


In [3]:
# Per-wallet total incidence (input + output, deduped on tx)
all_incidence = defaultdict(set)
for w, lst in wallet_in_txs.items():
    for tx, _ in lst: all_incidence[w].add(tx)
for w, lst in wallet_out_txs.items():
    for tx, _ in lst: all_incidence[w].add(tx)
wallet_deg = np.array([len(s) for s in all_incidence.values()])

print("Wallet incidence-degree distribution (txs per wallet):")
print(f"  total wallets:          {len(wallet_deg):,}")
print(f"  min / median / mean / max:  {wallet_deg.min()} / {int(np.median(wallet_deg))} / {wallet_deg.mean():.2f} / {wallet_deg.max()}")
for q in [50, 75, 90, 95, 99, 99.9, 99.99]:
    print(f"  p{q}:  {int(np.percentile(wallet_deg, q))}")

print("\n  count of wallets at each degree threshold (cumulative ≥):")
for thr in [2, 5, 10, 20, 50, 100, 500, 1000, 5000]:
    n = int((wallet_deg >= thr).sum())
    print(f"    deg ≥ {thr:>5d}:  {n:>7,}  wallets  ({100*n/len(wallet_deg):.2f}%)")

Wallet incidence-degree distribution (txs per wallet):
  total wallets:          822,942
  min / median / mean / max:  1 / 1 / 1.54 / 1471
  p50:  1
  p75:  2
  p90:  2
  p95:  3
  p99:  6
  p99.9:  20
  p99.99:  149

  count of wallets at each degree threshold (cumulative ≥):
    deg ≥     2:  273,268  wallets  (33.21%)
    deg ≥     5:   14,314  wallets  (1.74%)
    deg ≥    10:    2,541  wallets  (0.31%)
    deg ≥    20:      845  wallets  (0.10%)
    deg ≥    50:      263  wallets  (0.03%)
    deg ≥   100:      117  wallets  (0.01%)
    deg ≥   500:       19  wallets  (0.00%)
    deg ≥  1000:        4  wallets  (0.00%)
    deg ≥  5000:        0  wallets  (0.00%)


In [4]:
# Estimated cross-step tx->tx edge count for various hub-pruning caps.
# A wallet w with k incident txs contributes binom(k,2) ordered cross-step edges
# (assuming all txs at distinct timesteps; in practice some are at the same
# timestep — those are skipped because we require t(T_i) < t(T_j) strictly).
# For an upper bound we use k*(k-1)/2 here; we'll measure the exact count
# in the next cell after actually constructing the edges at one cap.

def estimated_edges_at_cap(cap):
    total = 0
    for w, txs in all_incidence.items():
        k = min(len(txs), cap)
        if k >= 2:
            total += k*(k-1)//2
    return total

print("Upper-bound cross-step edge count at each per-wallet cap:")
print(f"  {'cap':>6s}  {'wallets used':>14s}  {'est. edges (k(k-1)/2 sum)':>30s}")
for cap in [10, 20, 50, 100, 200, 500, 1000, 5000, 10**9]:
    n_used = sum(1 for txs in all_incidence.values() if len(txs) >= 2)
    e = estimated_edges_at_cap(cap)
    print(f"  {cap:>6d}  {n_used:>14,}  {e:>30,}")

Upper-bound cross-step edge count at each per-wallet cap:
     cap    wallets used       est. edges (k(k-1)/2 sum)


      10         273,268                         666,155


      20         273,268                         842,781


      50         273,268                       1,263,167


     100         273,268                       1,884,526
     200         273,268                       3,064,382


     500         273,268                       5,879,855
    1000         273,268                      10,621,528


    5000         273,268                      11,583,627


  1000000000         273,268                      11,583,627


In [5]:
# Build the actual cross-step tx->tx edge set at MAX_WALLET_DEGREE = 50.
# This is conservative; we'll explore alternative caps in the next cell.
MAX_WALLET_DEGREE = 50

print(f"Building cross-step tx→tx edges with wallet-degree cap = {MAX_WALLET_DEGREE}...")
t0 = time.time()

src, dst, dt_arr, role_arr, w_arr = [], [], [], [], []
ROLE_OUT_IN  = 0  # T_i.output_w == T_j.input_w   (money flow)
ROLE_OUT_OUT = 1
ROLE_IN_IN   = 2
ROLE_IN_OUT  = 3

# For each wallet, gather all (tx, t, side) events sorted by t,
# then for each ordered pair emit an edge with the role determined by sides.
wallets_skipped_hub = 0
for w in all_incidence:
    if len(all_incidence[w]) < 2:
        continue
    if len(all_incidence[w]) > MAX_WALLET_DEGREE:
        wallets_skipped_hub += 1
        continue
    # Collect events with side info
    events = []
    for tx, t in wallet_in_txs.get(w, []):
        events.append((t, tx, 'in'))   # w was input to tx (tx received from w)
    for tx, t in wallet_out_txs.get(w, []):
        events.append((t, tx, 'out'))  # w was output of tx (tx sent to w)
    events.sort(key=lambda r: r[0])
    K = len(events)
    for i in range(K):
        ti, txi, si = events[i]
        for j in range(i+1, K):
            tj, txj, sj = events[j]
            if tj <= ti:
                continue   # require strict t(T_i) < t(T_j)
            if txi == txj:
                continue   # same tx (a wallet can be both input and output of one tx)
            # role: side of T_i / side of T_j
            if si == 'out' and sj == 'in':   role = ROLE_OUT_IN
            elif si == 'out' and sj == 'out': role = ROLE_OUT_OUT
            elif si == 'in'  and sj == 'in':  role = ROLE_IN_IN
            else:                              role = ROLE_IN_OUT
            src.append(txi); dst.append(txj)
            dt_arr.append(tj - ti); role_arr.append(role); w_arr.append(w)

src      = np.array(src, dtype=np.int64)
dst      = np.array(dst, dtype=np.int64)
dt_edges = np.array(dt_arr, dtype=np.int32)
role_e   = np.array(role_arr, dtype=np.int8)
w_edges  = np.array(w_arr, dtype=np.int64)

E = len(src)
print(f"  built {E:,} cross-step edges in {time.time()-t0:.1f}s")
print(f"  wallets skipped (deg > {MAX_WALLET_DEGREE}): {wallets_skipped_hub:,}")

Building cross-step tx→tx edges with wallet-degree cap = 50...


  built 445,180 cross-step edges in 1.2s
  wallets skipped (deg > 50): 255


In [6]:
# Edge composition
print("Edges by role:")
for r, name in [(0,"out→in (money flow)"), (1,"out→out"), (2,"in→in"), (3,"in→out")]:
    n = int((role_e == r).sum())
    print(f"  {name:25s}: {n:>10,}  ({100*n/E:.2f}%)")

print("\nEdges by Δt bucket:")
for lo, hi in [(1,1),(2,2),(3,5),(6,10),(11,20),(21,49)]:
    m = (dt_edges >= lo) & (dt_edges <= hi)
    print(f"  Δt ∈ [{lo:>2d},{hi:>2d}]:  {int(m.sum()):>10,}  ({100*m.sum()/E:.2f}%)")

# In-degree of each tx in the cross-step graph (how many past txs feed it)
in_deg = np.bincount(dst, minlength=N_tx)
print(f"\nIn-degree (incoming cross-step edges per tx):")
print(f"  txs with ≥1 incoming edge: {int((in_deg>0).sum()):,}/{N_tx:,}  "
      f"({100*(in_deg>0).sum()/N_tx:.1f}%)")
print(f"  median / mean / max in-degree: {int(np.median(in_deg))} / {in_deg.mean():.1f} / {in_deg.max()}")
for q in [50, 75, 90, 95, 99, 99.9]:
    print(f"  p{q}:  {int(np.percentile(in_deg, q))}")

# Same restricted to LABELED txs (where this matters)
lab_mask = (tx_label != -1)
in_deg_lab = in_deg[lab_mask]
print(f"\n  LABELED txs only (n={lab_mask.sum():,}):")
print(f"    txs with ≥1 incoming edge: {int((in_deg_lab>0).sum()):,}/{int(lab_mask.sum()):,}  "
      f"({100*(in_deg_lab>0).sum()/lab_mask.sum():.1f}%)")
print(f"    median / mean / max: {int(np.median(in_deg_lab))} / {in_deg_lab.mean():.1f} / {in_deg_lab.max()}")

# Coverage broken down by class
for cls, name in [(1,"illicit"), (0,"licit")]:
    m = (tx_label == cls)
    n = int(m.sum())
    cov = int((in_deg[m] > 0).sum())
    print(f"    {name:8s} (n={n:>5,}): coverage {cov:,}/{n:,} = {100*cov/n:.1f}% have ≥1 incoming edge")

Edges by role:
  out→in (money flow)      :     75,982  (17.07%)
  out→out                  :    214,667  (48.22%)
  in→in                    :     62,574  (14.06%)
  in→out                   :     91,957  (20.66%)

Edges by Δt bucket:
  Δt ∈ [ 1, 1]:      66,542  (14.95%)
  Δt ∈ [ 2, 2]:      37,279  (8.37%)
  Δt ∈ [ 3, 5]:      93,729  (21.05%)
  Δt ∈ [ 6,10]:      83,023  (18.65%)
  Δt ∈ [11,20]:     102,209  (22.96%)
  Δt ∈ [21,49]:      62,398  (14.02%)

In-degree (incoming cross-step edges per tx):
  txs with ≥1 incoming edge: 31,989/203,769  (15.7%)
  median / mean / max in-degree: 0 / 2.2 / 12882
  p50:  0
  p75:  0
  p90:  2
  p95:  6
  p99:  40
  p99.9:  151

  LABELED txs only (n=46,564):
    txs with ≥1 incoming edge: 12,292/46,564  (26.4%)
    median / mean / max: 0 / 4.5 / 12882
    illicit  (n=4,545): coverage 210/4,545 = 4.6% have ≥1 incoming edge
    licit    (n=42,019): coverage 12,082/42,019 = 28.8% have ≥1 incoming edge


In [7]:
# Sanity: verify causality. For every edge T_i -> T_j we must have t(T_i) < t(T_j).
src_t = tx_time[src]
dst_t = tx_time[dst]
n_violations = int((src_t >= dst_t).sum())
print(f"Causality check: {n_violations} edges with t(src) >= t(dst)  (must be 0)")
assert n_violations == 0

# Distribution of the strongest signal: incoming edges from ILLICIT past txs to each tx
src_lab = tx_label[src]
illicit_src_mask = (src_lab == 1)
in_deg_from_illicit = np.bincount(dst[illicit_src_mask], minlength=N_tx)
print(f"\nIncoming edges from ILLICIT past txs:")
print(f"  total such edges: {int(illicit_src_mask.sum()):,}")
print(f"  txs with ≥1 illicit predecessor: {int((in_deg_from_illicit>0).sum()):,}")
for cls, name in [(1,"illicit"), (0,"licit"), (-1,"unknown")]:
    m = (tx_label == cls)
    n = int(m.sum())
    cov = int((in_deg_from_illicit[m] > 0).sum())
    rate = 100*cov/n if n else 0
    print(f"    among {name:8s} ({n:>6,} txs):  {cov:>5,} ({rate:5.2f}%) have ≥1 illicit predecessor")

# Same restricted to test split (t > TRAIN_END)
test_mask = (tx_time > TRAIN_END) & (tx_label != -1)
for cls, name in [(1,"illicit"), (0,"licit")]:
    m = test_mask & (tx_label == cls)
    n = int(m.sum())
    cov = int((in_deg_from_illicit[m] > 0).sum())
    rate = 100*cov/n if n else 0
    print(f"    TEST {name:8s} ({n:>5,} txs):  {cov:>4,} ({rate:5.2f}%) have ≥1 illicit predecessor")

Causality check: 0 edges with t(src) >= t(dst)  (must be 0)

Incoming edges from ILLICIT past txs:
  total such edges: 759
  txs with ≥1 illicit predecessor: 369
    among illicit  ( 4,545 txs):    108 ( 2.38%) have ≥1 illicit predecessor
    among licit    (42,019 txs):     68 ( 0.16%) have ≥1 illicit predecessor
    among unknown  (157,205 txs):    193 ( 0.12%) have ≥1 illicit predecessor
    TEST illicit  (1,083 txs):    21 ( 1.94%) have ≥1 illicit predecessor
    TEST licit    (15,587 txs):    26 ( 0.17%) have ≥1 illicit predecessor


In [8]:
# Sweep MAX_WALLET_DEGREE to see how edge count and labeled-tx coverage trade off.
def build_edges_at_cap(cap, max_edges=None):
    src_l, dst_l = [], []
    for w in all_incidence:
        if len(all_incidence[w]) < 2 or len(all_incidence[w]) > cap:
            continue
        events = []
        for tx, t in wallet_in_txs.get(w, []):  events.append((t, tx))
        for tx, t in wallet_out_txs.get(w, []): events.append((t, tx))
        events.sort(key=lambda r: r[0])
        K = len(events)
        for i in range(K):
            ti, txi = events[i]
            for j in range(i+1, K):
                tj, txj = events[j]
                if tj > ti and txi != txj:
                    src_l.append(txi); dst_l.append(txj)
                    if max_edges and len(src_l) > max_edges:
                        return None, None, True
    return np.array(src_l, dtype=np.int64), np.array(dst_l, dtype=np.int64), False

print(f"{'cap':>6s}  {'edges':>14s}  {'labeled coverage':>18s}  {'illicit-pred. rate (test illicit)':>40s}")
for cap in [10, 20, 50, 100, 200]:
    s, d, overflow = build_edges_at_cap(cap, max_edges=80_000_000)
    if overflow:
        print(f"  {cap:>4d}  >80M edges (skipped)")
        continue
    ind = np.bincount(d, minlength=N_tx)
    src_t_local = tx_time[s]; dst_t_local = tx_time[d]
    src_lab_local = tx_label[s]
    ill_src = (src_lab_local == 1)
    ind_ill = np.bincount(d[ill_src], minlength=N_tx)
    lab_cov = int((ind[tx_label != -1] > 0).sum())
    lab_n   = int((tx_label != -1).sum())
    test_ill_mask = (tx_time > TRAIN_END) & (tx_label == 1)
    n_test_ill = int(test_ill_mask.sum())
    cov_test_ill = int((ind_ill[test_ill_mask] > 0).sum())
    rate = 100*cov_test_ill/max(n_test_ill,1)
    print(f"  {cap:>4d}  {len(s):>14,}  {lab_cov:>6,}/{lab_n:<6,} ({100*lab_cov/lab_n:5.1f}%)   "
          f"{cov_test_ill:>4,}/{n_test_ill:<4,} ({rate:5.2f}%)")

   cap           edges    labeled coverage         illicit-pred. rate (test illicit)


    10         227,495   9,391/46,564 ( 20.2%)     17/1,083 ( 1.57%)


    20         281,808  10,480/46,564 ( 22.5%)     21/1,083 ( 1.94%)


    50         445,180  12,292/46,564 ( 26.4%)     21/1,083 ( 1.94%)


   100         683,825  12,691/46,564 ( 27.3%)     21/1,083 ( 1.94%)


   200       1,106,272  13,266/46,564 ( 28.5%)     21/1,083 ( 1.94%)


In [9]:
print("=" * 70)
print("STEP 1 SUMMARY")
print("=" * 70)
print(f"  Cross-step tx→tx edges built at cap={MAX_WALLET_DEGREE}: E = {E:,}")
print(f"  Coverage of labeled txs with ≥1 incoming cross-step edge: see above.")
print()
print("  Decisions to make in step 2 (feature engineering):")
print("    - Final wallet-degree cap (the sweep above informs this).")
print("    - Whether to also build a *role-restricted* graph (out→in only) since that's the")
print("      true money-flow signal; the other 3 roles may dilute.")
print("    - Whether to weight edges by 1/wallet_deg (penalize high-degree shared wallets).")
print()
print("  Variables left in scope for step 2:")
print("    src, dst, dt_edges, role_e, w_edges  (the edge tensors)")
print("    in_deg, in_deg_from_illicit          (per-tx in-degree summaries)")
print("    incident_in_w, incident_out_w        (bipartite, for cross-checks)")
print("    tx_time, tx_label                    (per-tx ground truth)")

STEP 1 SUMMARY
  Cross-step tx→tx edges built at cap=50: E = 445,180
  Coverage of labeled txs with ≥1 incoming cross-step edge: see above.

  Decisions to make in step 2 (feature engineering):
    - Final wallet-degree cap (the sweep above informs this).
    - Whether to also build a *role-restricted* graph (out→in only) since that's the
      true money-flow signal; the other 3 roles may dilute.
    - Whether to weight edges by 1/wallet_deg (penalize high-degree shared wallets).

  Variables left in scope for step 2:
    src, dst, dt_edges, role_e, w_edges  (the edge tensors)
    in_deg, in_deg_from_illicit          (per-tx in-degree summaries)
    incident_in_w, incident_out_w        (bipartite, for cross-checks)
    tx_time, tx_label                    (per-tx ground truth)
